# 🔧 GUIDE DE CONFIGURATION DEVOPS - GMS

## 1️⃣ APPLICATIONS À TÉLÉCHARGER

### Obligatoires

| Outil | Utilité | Lien de téléchargement |
|-------|---------|------------------------|
| **Git** | Gestion de version | https://git-scm.com/downloads |
| **Docker Desktop** | Conteneurisation | https://www.docker.com/products/docker-desktop/ |
| **Visual Studio 2022** | IDE principal (.NET) | https://visualstudio.microsoft.com/fr/downloads/ |
| **.NET 8 SDK** | Framework de développement | https://dotnet.microsoft.com/download/dotnet/8.0 |
| **SQL Server Express** | Base de données locale | https://www.microsoft.com/fr-fr/sql-server/sql-server-downloads |
| **SQL Server Management Studio** | Gestion BDD | https://aka.ms/ssmsfullsetup |

### Recommandés

| Outil | Utilité | Lien |
|-------|---------|------|
| **Visual Studio Code** | Éditeur léger | https://code.visualstudio.com/ |
| **Postman** | Tests API | https://www.postman.com/downloads/ |
| **Azure Data Studio** | Client BDD moderne | https://docs.microsoft.com/sql/azure-data-studio/download |
| **GitKraken** | Client Git visuel | https://www.gitkraken.com/download |
| **DBeaver** | Client BDD universel | https://dbeaver.io/download/ |

### CI/CD (Optionnel pour MVP)

| Outil | Utilité | Lien |
|-------|---------|------|
| **GitHub Desktop** | Interface Git simplifiée | https://desktop.github.com/ |
| **Jenkins** | Automatisation CI/CD | https://www.jenkins.io/download/ |
| **TeamCity** | CI/CD JetBrains | https://www.jetbrains.com/teamcity/ |

---

## 2️⃣ INSTALLATION & CONFIGURATION INITIALE

### A. Installer Git

```bash
# Windows : Exécuter l'installeur
# Vérifier l'installation
git --version

# Configuration initiale
git config --global user.name "Votre Nom"
git config --global user.email "votre.email@exemple.com"
git config --global init.defaultBranch main
```

### B. Installer Docker Desktop

```bash
# Windows : Installer Docker Desktop
# Activer WSL 2 (Windows Subsystem for Linux)
# Vérifier l'installation
docker --version
docker-compose --version

# Tester Docker
docker run hello-world
```

### C. Installer .NET 8 SDK

```bash
# Vérifier l'installation
dotnet --version
# Devrait afficher : 8.0.x

# Installer les outils EF Core globalement
dotnet tool install --global dotnet-ef
dotnet tool update --global dotnet-ef
```

### D. Installer SQL Server Express

```bash
# Après installation, vérifier le service
# Services Windows → SQL Server (SQLEXPRESS) → Démarré

# Chaîne de connexion par défaut :
Server=(localdb)\mssqllocaldb;Database=GmsDb;Trusted_Connection=True;
```

---

## 3️⃣ STRUCTURE DU PROJET GIT

### A. Initialiser le repository

```bash
# Créer le dossier projet
mkdir GMS-School-Management
cd GMS-School-Management

# Initialiser Git
git init

# Créer le .gitignore
```

### B. Structure des dossiers

```
GMS-School-Management/
├── .github/
│   └── workflows/              # GitHub Actions CI/CD
│       ├── build.yml
│       └── release.yml
├── docker/
│   ├── Dockerfile.api
│   ├── Dockerfile.desktop
│   └── docker-compose.yml
├── docs/
│   ├── cahier-charges.md
│   ├── architecture.md
│   ├── api-documentation.md
│   └── user-guide.md
├── scripts/
│   ├── setup-dev.ps1           # Script config environnement
│   ├── migrate-database.ps1
│   └── backup-database.ps1
├── src/
│   ├── GMS.Domain/
│   ├── GMS.Application/
│   ├── GMS.Infrastructure/
│   ├── GMS.Desktop/
│   ├── GMS.Shared/
│   └── GMS.Tests/
├── .gitignore
├── .dockerignore
├── README.md
├── docker-compose.yml
└── GMS.sln
```

---

## 4️⃣ FICHIERS DE CONFIGURATION DEVOPS

### A. .gitignore (Pour .NET)

```gitignore
# Build results
[Dd]ebug/
[Rr]elease/
x64/
x86/
[Bb]in/
[Oo]bj/

# Visual Studio
.vs/
*.user
*.suo
*.userosscache
*.sln.docstates

# User-specific files
*.rsuser
*.suo
*.user
*.userosscache
*.sln.docstates

# Test Results
[Tt]est[Rr]esult*/
[Bb]uild[Ll]og.*

# NuGet Packages
*.nupkg
*.snupkg
**/packages/*
!**/packages/build/

# Database
*.mdf
*.ldf
*.ndf

# SQL Server files
*.bak
*.bakup

# User secrets
appsettings.Development.json
appsettings.Production.json
**/appsettings.*.json
!**/appsettings.json

# Docker
**/docker-compose.override.yml

# Logs
logs/
*.log

# Photos/uploads
**/wwwroot/uploads/
**/Photos/
```

### B. docker-compose.yml

```yaml
version: '3.8'

services:
  sqlserver:
    image: mcr.microsoft.com/mssql/server:2022-latest
    container_name: gms-sqlserver
    environment:
      - ACCEPT_EULA=Y
      - SA_PASSWORD=GmsP@ssw0rd2024!
      - MSSQL_PID=Express
    ports:
      - "1433:1433"
    volumes:
      - sqlserver-data:/var/opt/mssql
      - ./scripts/init-db.sql:/docker-entrypoint-initdb.d/init-db.sql
    networks:
      - gms-network
    restart: unless-stopped

  gms-desktop:
    build:
      context: .
      dockerfile: docker/Dockerfile.desktop
    container_name: gms-desktop-app
    depends_on:
      - sqlserver
    environment:
      - ConnectionStrings__DefaultConnection=Server=sqlserver;Database=GmsDb;User=sa;Password=GmsP@ssw0rd2024!;TrustServerCertificate=True
    networks:
      - gms-network

volumes:
  sqlserver-data:
    driver: local

networks:
  gms-network:
    driver: bridge
```

### C. Dockerfile.desktop

```dockerfile
# Stage 1: Build
FROM mcr.microsoft.com/dotnet/sdk:8.0 AS build
WORKDIR /src

# Copier les fichiers de solution
COPY ["src/GMS.Desktop/GMS.Desktop.csproj", "GMS.Desktop/"]
COPY ["src/GMS.Application/GMS.Application.csproj", "GMS.Application/"]
COPY ["src/GMS.Infrastructure/GMS.Infrastructure.csproj", "GMS.Infrastructure/"]
COPY ["src/GMS.Domain/GMS.Domain.csproj", "GMS.Domain/"]
COPY ["src/GMS.Shared/GMS.Shared.csproj", "GMS.Shared/"]

# Restaurer les dépendances
RUN dotnet restore "GMS.Desktop/GMS.Desktop.csproj"

# Copier tout le code
COPY src/ .

# Build
WORKDIR "/src/GMS.Desktop"
RUN dotnet build "GMS.Desktop.csproj" -c Release -o /app/build

# Stage 2: Publish
FROM build AS publish
RUN dotnet publish "GMS.Desktop.csproj" -c Release -o /app/publish /p:UseAppHost=true

# Stage 3: Runtime
FROM mcr.microsoft.com/dotnet/runtime:8.0
WORKDIR /app
COPY --from=publish /app/publish .

ENTRYPOINT ["dotnet", "GMS.Desktop.dll"]
```

### D. .dockerignore

```dockerignore
**/.git
**/.gitignore
**/.vs
**/.vscode
**/bin
**/obj
**/.dockerignore
**/Dockerfile*
**/docker-compose*
**/*.md
**/charts
**/.env
**/secrets.json
```

---

## 5️⃣ SCRIPTS AUTOMATISATION

### A. setup-dev.ps1 (Windows PowerShell)

```powershell
# Script de configuration environnement développement
Write-Host "🚀 Configuration environnement GMS..." -ForegroundColor Green

# Vérifier les prérequis
Write-Host "Vérification des outils installés..." -ForegroundColor Yellow

# Git
if (Get-Command git -ErrorAction SilentlyContinue) {
    Write-Host "✅ Git installé : $(git --version)" -ForegroundColor Green
} else {
    Write-Host "❌ Git non installé" -ForegroundColor Red
    exit 1
}

# .NET
if (Get-Command dotnet -ErrorAction SilentlyContinue) {
    Write-Host "✅ .NET installé : $(dotnet --version)" -ForegroundColor Green
} else {
    Write-Host "❌ .NET SDK non installé" -ForegroundColor Red
    exit 1
}

# Docker
if (Get-Command docker -ErrorAction SilentlyContinue) {
    Write-Host "✅ Docker installé : $(docker --version)" -ForegroundColor Green
} else {
    Write-Host "❌ Docker non installé" -ForegroundColor Red
    exit 1
}

# Restaurer les packages NuGet
Write-Host "📦 Restauration des packages NuGet..." -ForegroundColor Yellow
dotnet restore

# Créer la base de données
Write-Host "🗄️ Création de la base de données..." -ForegroundColor Yellow
dotnet ef database update --project src/GMS.Infrastructure --startup-project src/GMS.Desktop

# Seed des données
Write-Host "🌱 Insertion des données de test..." -ForegroundColor Yellow
# Le seed se fait automatiquement au démarrage

# Build du projet
Write-Host "🔨 Build du projet..." -ForegroundColor Yellow
dotnet build

Write-Host "✅ Configuration terminée avec succès!" -ForegroundColor Green
Write-Host "Lancez l'application avec : dotnet run --project src/GMS.Desktop" -ForegroundColor Cyan
```

### B. migrate-database.ps1

```powershell
# Script de migration base de données
param(
    [string]$MigrationName = "NewMigration"
)

Write-Host "🗄️ Création de la migration : $MigrationName" -ForegroundColor Yellow

# Créer la migration
dotnet ef migrations add $MigrationName `
    --project src/GMS.Infrastructure `
    --startup-project src/GMS.Desktop `
    --output-dir Data/Migrations

# Appliquer la migration
Write-Host "📝 Application de la migration..." -ForegroundColor Yellow
dotnet ef database update `
    --project src/GMS.Infrastructure `
    --startup-project src/GMS.Desktop

Write-Host "✅ Migration appliquée avec succès!" -ForegroundColor Green
```

---

## 6️⃣ WORKFLOW CI/CD GITHUB ACTIONS

### .github/workflows/build.yml

```yaml
name: Build & Test

on:
  push:
    branches: [ main, develop ]
  pull_request:
    branches: [ main, develop ]

jobs:
  build:
    runs-on: windows-latest

    steps:
    - uses: actions/checkout@v3
    
    - name: Setup .NET
      uses: actions/setup-dotnet@v3
      with:
        dotnet-version: 8.0.x
    
    - name: Restore dependencies
      run: dotnet restore
    
    - name: Build
      run: dotnet build --no-restore --configuration Release
    
    - name: Test
      run: dotnet test --no-build --configuration Release --verbosity normal
    
    - name: Publish
      run: dotnet publish src/GMS.Desktop/GMS.Desktop.csproj -c Release -o ./publish
    
    - name: Upload artifact
      uses: actions/upload-artifact@v3
      with:
        name: gms-desktop-build
        path: ./publish
```

---

## 7️⃣ COMMANDES GIT ESSENTIELLES

### Workflow quotidien

```bash
# 1. Créer une nouvelle branche feature
git checkout -b feature/gestion-eleves

# 2. Faire des modifications et les ajouter
git add .
git status

# 3. Commit avec message descriptif
git commit -m "feat: Ajout module inscription élèves"

# 4. Pousser vers le repository distant
git push origin feature/gestion-eleves

# 5. Créer une Pull Request sur GitHub
# (via interface web)

# 6. Après merge, revenir sur main et mettre à jour
git checkout main
git pull origin main

# 7. Supprimer la branche locale
git branch -d feature/gestion-eleves
```

### Conventions de commit (Conventional Commits)

```bash
feat: Nouvelle fonctionnalité
fix: Correction de bug
docs: Documentation
style: Formatage, indentation
refactor: Refactorisation sans changement fonctionnel
test: Ajout/modification tests
chore: Maintenance, configuration

# Exemples :
git commit -m "feat: Ajout authentification utilisateur"
git commit -m "fix: Correction calcul solde famille"
git commit -m "docs: Mise à jour README avec instructions Docker"
```

---

## 8️⃣ LANCEMENT DU PROJET

### Option 1 : Développement local (Sans Docker)

```bash
# 1. Cloner le repository
git clone https://github.com/votre-username/GMS-School-Management.git
cd GMS-School-Management

# 2. Configurer l'environnement
.\scripts\setup-dev.ps1

# 3. Lancer l'application
dotnet run --project src/GMS.Desktop
```

### Option 2 : Avec Docker

```bash
# 1. Démarrer les conteneurs
docker-compose up -d

# 2. Vérifier les logs
docker-compose logs -f

# 3. Accéder à SQL Server
# Host: localhost
# Port: 1433
# User: sa
# Password: GmsP@ssw0rd2024!

# 4. Arrêter les conteneurs
docker-compose down
```

### Option 3 : Build production

```bash
# Build optimisé pour production
dotnet publish src/GMS.Desktop/GMS.Desktop.csproj `
    -c Release `
    -r win-x64 `
    --self-contained true `
    -p:PublishSingleFile=true `
    -p:PublishTrimmed=true `
    -o ./release

# L'exécutable sera dans ./release/GMS.Desktop.exe
```

---

## 9️⃣ GESTION DES ENVIRONNEMENTS

### appsettings.json (Développement)

```json
{
  "ConnectionStrings": {
    "DefaultConnection": "Server=(localdb)\\mssqllocaldb;Database=GmsDb;Trusted_Connection=True;"
  },
  "Logging": {
    "LogLevel": {
      "Default": "Information",
      "Microsoft": "Warning"
    }
  }
}
```

### appsettings.Production.json

```json
{
  "ConnectionStrings": {
    "DefaultConnection": "Server=prod-server;Database=GmsDb;User=gms_user;Password=***;Encrypt=True;"
  },
  "Logging": {
    "LogLevel": {
      "Default": "Warning",
      "Microsoft": "Error"
    }
  }
}
```

---

## 🔟 BACKUP & RESTAURATION

### Script backup automatique

```powershell
# backup-database.ps1
$date = Get-Date -Format "yyyyMMdd_HHmmss"
$backupPath = "C:\GMS\Backups\GmsDb_$date.bak"

sqlcmd -S (localdb)\mssqllocaldb -Q "BACKUP DATABASE GmsDb TO DISK='$backupPath'"

Write-Host "✅ Backup créé : $backupPath" -ForegroundColor Green
```

---

## ✅ CHECKLIST DÉMARRAGE PROJET

- [ ] Git installé et configuré
- [ ] Docker Desktop installé
- [ ] .NET 8 SDK installé
- [ ] Visual Studio 2022 installé
- [ ] SQL Server Express installé
- [ ] Repository Git initialisé
- [ ] Docker Compose testé
- [ ] Base de données créée
- [ ] Données de seed insérées
- [ ] Premier build réussi
- [ ] Documentation lue

---

**Prêt pour développer ! 🚀**